In [0]:
# Databricks notebook source
%pip install dbldatagen Faker

# COMMAND ----------

dbutils.library.restartPython()

In [0]:
# Widgets (also work as Job parameters)
def get_widget(name, default, choices=None):
    if choices:
        dbutils.widgets.dropdown(name, default, choices)
    else:
        dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name).strip()

from data_generator import PRESETS

PRESET      = get_widget("preset", "small", list(PRESETS))
W_CUSTOMERS = get_widget("customers", "")
W_PRODUCTS  = get_widget("products", "")
W_CATEGORIES= get_widget("categories", "")
W_ORDERS    = get_widget("orders", "")
SEED        = int(get_widget("seed", "42") or 42)
OUTPUT_MODE = get_widget("output_mode", "temp_views", ["temp_views", "delta"])
CATALOG     = get_widget("catalog", "main")
SCHEMA      = get_widget("schema", "synthetic_ecommerce")
RUN_CHECKS  = get_widget("run_checks", "true", ["true", "false"]) == "true"

# COMMAND ----------

# Build the config (blank widget = use the preset value)
from data_generator import (
    build_config, parse_int, print_row_plan, generate_all_tables,
    publish_tables, run_checks, show,
)

cfg = build_config(
    preset=PRESET,
    customers=parse_int(W_CUSTOMERS, None) if W_CUSTOMERS else None,
    products=parse_int(W_PRODUCTS, None) if W_PRODUCTS else None,
    categories=parse_int(W_CATEGORIES, None) if W_CATEGORIES else None,
    orders=parse_int(W_ORDERS, None) if W_ORDERS else None,
    seed=SEED,
    # optional overrides, e.g.:
    # ratios=dict(max_items_per_order=4, return_rate=0.15),
    # locales=[("GB", "en_GB", 70), ("US", "en_US", 30)],
    # currency="USD",
)
print_row_plan(cfg, OUTPUT_MODE)

# COMMAND ----------

# Generate (lazy: nothing runs until an action)
tables = generate_all_tables(spark, cfg)

# COMMAND ----------

# Publish as temp views or Delta tables
tables = publish_tables(spark, tables, OUTPUT_MODE, CATALOG, SCHEMA)

# COMMAND ----------

# Sanity checks (set run_checks=false to skip on large volumes)
if RUN_CHECKS:
    run_checks(tables, cfg)

# COMMAND ----------

# Peek at a few tables
for t in ["customer", "address", "product", "order", "order_item", "return_item"]:
    print(t)
    show(tables[t], 8)

# COMMAND ----------

# Test the functions independently: one table at a time with a tiny config
from data_generator import generate_customers, generate_products, generate_order_items

tiny = build_config(preset="tiny", seed=1)
print(generate_customers(spark, tiny).count(), "customers")
print(generate_products(spark, tiny).count(), "products")
print(generate_order_items(spark, tiny).count(), "order items")